# MeloTTS Vietnamese - Text-to-Speech

## What is MeloTTS?

[MeloTTS](https://github.com/myshell-ai/MeloTTS) is an open-source, high-quality text-to-speech library developed by **MyShell AI**. It is built on **VITS2** (Variational Inference with adversarial learning for end-to-end Text-to-Speech, version 2) combined with **BERT** for improved prosody and naturalness.

### Key Facts

- **Architecture**: VITS2 + BERT-based prosody modeling
- **Official languages**: English (EN), Spanish (ES), French (FR), Chinese (ZH), Japanese (JA), Korean (KO)
- **Vietnamese**: NOT officially supported. A community port exists:
  - GitHub: [manhcuong02/MeloTTS_Vietnamese](https://github.com/manhcuong02/MeloTTS_Vietnamese)
  - HuggingFace: [nmcuong/MeloTTS-Vietnamese](https://huggingface.co/nmcuong/MeloTTS-Vietnamese)
- **Voice cloning**: Not supported. MeloTTS is multi-speaker but **not** zero-shot — it cannot clone arbitrary voices from a reference audio clip
- **License**: MIT
- **Speed**: Real-time or faster inference on GPU thanks to the VITS2 architecture (single-stage, no vocoder needed)

### How VITS2 Works (simplified)

VITS2 is an end-to-end TTS model that combines:
1. A text encoder (with BERT embeddings for prosody)
2. A stochastic duration predictor
3. A flow-based decoder that directly outputs waveforms

This means there is no separate vocoder step — the model goes directly from text to audio, which makes it both fast and high-quality.

## 1. Installation

We install the Vietnamese community fork of MeloTTS from GitHub. This fork adds Vietnamese language support including phonemization, BERT models, and pretrained checkpoints.

In [ ]:
# Install build tools and missing deps
!pip install -q maturin setuptools-rust g2p_en

# Clone the Vietnamese fork
!git clone -q https://github.com/manhcuong02/MeloTTS_Vietnamese

# Install requirements
!cd MeloTTS_Vietnamese && pip install -q -r requirements.txt 2>&1 | tail -3

# Add to Python path (avoids broken pip install -e build)
import sys
sys.path.insert(0, '/kaggle/working/MeloTTS_Vietnamese')

# Verify import
from melo.api import TTS
print("MeloTTS Vietnamese loaded successfully!")

## 2. Load the Vietnamese TTS Model

MeloTTS provides a simple `TTS` class. We specify `language='VI'` to load the Vietnamese model. On first run, the pretrained checkpoint and BERT model will be downloaded automatically from HuggingFace.

In [ ]:
from melo.api import TTS

# Load the Vietnamese TTS model
# device='auto' will use GPU if available, otherwise CPU
tts = TTS(language='VI', device='auto')

# List available speaker IDs
print("Available speakers:", tts.hps.data.spk2id)

## 3. Basic Vietnamese TTS

Let's generate speech from a simple Vietnamese greeting. The `tts_to_file` method takes:
- **text**: The Vietnamese text to synthesize
- **speaker_id**: The speaker identity (from `spk2id`)
- **output_path**: Where to save the WAV file
- **speed**: Speaking rate (1.0 = normal)

In [ ]:
import IPython.display as ipd

# Get the speaker ID for Vietnamese
speaker_ids = tts.hps.data.spk2id
speaker_id = list(speaker_ids.values())[0]  # Use the first available speaker
print(f"Using speaker ID: {speaker_id}")

# Simple greeting
text = "Xin chào, tôi là trợ lý ảo. Rất vui được gặp bạn."
output_path = "output_hello.wav"

tts.tts_to_file(text, speaker_id, output_path, speed=1.0)
print(f"Saved to {output_path}")
ipd.display(ipd.Audio(output_path))

## 4. More Vietnamese Examples

Let's try different types of Vietnamese text to see how the model handles various scenarios: tones, questions, longer passages, and numbers.

In [ ]:
# Example 1: A question with Vietnamese tones
text_question = "Bạn có khỏe không? Hôm nay thời tiết thế nào?"
output_question = "output_question.wav"

tts.tts_to_file(text_question, speaker_id, output_question, speed=1.0)
print(f"Question: {text_question}")
ipd.display(ipd.Audio(output_question))

In [ ]:
# Example 2: A longer passage - Vietnamese proverb and explanation
text_passage = (
    "Học tập là con đường ngắn nhất để đạt được thành công. "
    "Người xưa có câu, có công mài sắt có ngày nên kim. "
    "Chúng ta hãy cùng nhau cố gắng mỗi ngày."
)
output_passage = "output_passage.wav"

tts.tts_to_file(text_passage, speaker_id, output_passage, speed=1.0)
print(f"Passage: {text_passage}")
ipd.display(ipd.Audio(output_passage))

In [ ]:
# Example 3: Text with numbers and common expressions
text_numbers = "Việt Nam có 54 dân tộc anh em và hơn 100 triệu người. Thủ đô Hà Nội đã có hơn 1000 năm lịch sử."
output_numbers = "output_numbers.wav"

tts.tts_to_file(text_numbers, speaker_id, output_numbers, speed=1.0)
print(f"Numbers: {text_numbers}")
ipd.display(ipd.Audio(output_numbers))

In [ ]:
# Example 4: Adjusting speaking speed
text_speed = "Tốc độ nói có thể được điều chỉnh nhanh hơn hoặc chậm hơn."

# Normal speed
tts.tts_to_file(text_speed, speaker_id, "output_speed_normal.wav", speed=1.0)
print("Normal speed (1.0):")
ipd.display(ipd.Audio("output_speed_normal.wav"))

# Slower
tts.tts_to_file(text_speed, speaker_id, "output_speed_slow.wav", speed=0.8)
print("Slow speed (0.8):")
ipd.display(ipd.Audio("output_speed_slow.wav"))

# Faster
tts.tts_to_file(text_speed, speaker_id, "output_speed_fast.wav", speed=1.3)
print("Fast speed (1.3):")
ipd.display(ipd.Audio("output_speed_fast.wav"))

## 5. Batch Processing Multiple Sentences

For generating audio for multiple sentences, we can loop through a list. This is useful for creating datasets or audiobook-style outputs.

In [ ]:
sentences = [
    "Trí tuệ nhân tạo đang thay đổi thế giới.",
    "Công nghệ chuyển văn bản thành giọng nói ngày càng tự nhiên.",
    "Tiếng Việt là ngôn ngữ có sáu thanh điệu.",
    "Chúc bạn một ngày tốt lành.",
]

for i, sentence in enumerate(sentences):
    output_file = f"output_batch_{i+1}.wav"
    tts.tts_to_file(sentence, speaker_id, output_file, speed=1.0)
    print(f"[{i+1}] {sentence}")
    ipd.display(ipd.Audio(output_file))

## 6. Key Takeaways

| Aspect | Details |
|---|---|
| **Architecture** | VITS2 + BERT — end-to-end, single-stage TTS (no separate vocoder) |
| **Speed** | Fast inference, real-time or better on GPU |
| **Quality** | Good naturalness for Vietnamese, handles tones well |
| **Voice cloning** | Not supported — multi-speaker but not zero-shot |
| **Vietnamese support** | Community-maintained fork, not official |
| **License** | MIT |

### Strengths
- **Fast**: VITS2 is a single-stage model, so inference is very efficient
- **Good quality**: BERT-based prosody modeling produces natural-sounding speech
- **Easy API**: Simple `tts_to_file()` interface with speed control
- **Open source**: MIT licensed, free to use commercially

### Limitations
- **No voice cloning**: Cannot replicate a target speaker's voice from a sample
- **Community-maintained**: Vietnamese support relies on the community fork, not the official repo
- **Limited speakers**: Only the pretrained speaker voices are available

### Links
- Official MeloTTS: https://github.com/myshell-ai/MeloTTS
- Vietnamese fork: https://github.com/manhcuong02/MeloTTS_Vietnamese
- HuggingFace model: https://huggingface.co/nmcuong/MeloTTS-Vietnamese